In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
#Read Bronze Tables
df_posts = spark.read.table("log_analytics.bronze.posts_raw")
df_comments = spark.read.table("log_analytics.bronze.comments_raw")

In [0]:
#Import functions
from pyspark.sql.functions import *
from pyspark.sql.types import *

#POSTS CLEANING

In [0]:
#Drop duplicates
df_posts = df_posts.dropDuplicates(["id"])

In [0]:
#Filter nulls
df_posts = df_posts.filter(
    col("userId").isNotNull() &
    col("id").isNotNull()
)

In [0]:
# Clean title
df_posts = df_posts.withColumn("title", trim(lower(col("title"))))

In [0]:
# Clean body
df_posts = df_posts.withColumn("body", trim(col("body")))

In [0]:
# Handle nulls(titles)
df_posts = df_posts.withColumn("title", when(col("title").isNull(), "unknown").otherwise(col("title")))

In [0]:
#Hindling nulls(body)
df_posts = df_posts.withColumn("body", when(col("body").isNull(), "no_content").otherwise(col("body")))

In [0]:
# Add features
df_posts = df_posts.withColumn("body_length", length(col("body")))

In [0]:
# Log level
df_posts = df_posts.withColumn(
    "log_level",
    when(col("body_length") > 200, "ERROR").otherwise("INFO")
)

In [0]:
# Error flag
df_posts = df_posts.withColumn(
    "is_error",
    when(col("log_level") == "ERROR", 1).otherwise(0)
)

In [0]:
# Add timestamp
df_posts = df_posts.withColumn("processed_time", current_timestamp())

In [0]:
df_posts = df_posts.select(
    "userId",              
    "id",
    "title",
    "body",
    "body_length",
    "log_level",
    "is_error",
    "ingestion_time",
    "processed_time"
)

In [0]:
#Write Posts Silver
try:
    spark.sql("DROP TABLE IF EXISTS log_analytics.silver.posts_logs")
except Exception:
    pass
df_posts.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true").saveAsTable("log_analytics.silver.posts_logs")

In [0]:
#drop duplicates
df_comments = df_comments.dropDuplicates(["id"])

In [0]:
df_comments = df_comments.filter(
    col("id").isNotNull() &
    col("postId").isNotNull()
)

In [0]:
#COMMENTS CLEANING
#Clean Name
df_comments = df_comments.withColumn("name", trim(lower(col("name"))))

In [0]:
#Clean Email
df_comments = df_comments.withColumn("email", trim(lower(col("email"))))

In [0]:
#Clean Body
df_comments = df_comments.withColumn("body", trim(col("body")))

In [0]:
df_comments = df_comments.withColumn(
    "email_valid",
    when(instr(col("email"), "@") > 0, "Valid").otherwise("Invalid Email")
)

In [0]:
#Add Activity Type
df_comments = df_comments.withColumn("activity_type", lit("comment"))

In [0]:
#Add Timestamp
df_comments = df_comments.withColumn("processed_time", current_timestamp())

In [0]:
# Select columns 
df_comments = df_comments.select(
    "id",
    "postId",
    "name",
    "email",
    "email_valid",  
    "body",
    "activity_type",
    "ingestion_time",
    "processed_time"
)

### INCREMENTAL FILTER

In [0]:
# Get max ingestion_time from Silver
if spark.catalog.tableExists("log_analytics.silver.posts_logs"):
    max_time = spark.sql("""
        SELECT max(ingestion_time) as max_time
        FROM log_analytics.silver.posts_logs
    """).collect()[0]["max_time"]

    df_posts_inc = df_posts.filter(col("ingestion_time") > lit(max_time))
else:
    df_posts_inc = df_posts

In [0]:
# COMMENTS
if spark.catalog.tableExists("log_analytics.silver.comments_logs"):

    max_time_c = spark.sql("""
        SELECT max(ingestion_time) as max_time
        FROM log_analytics.silver.comments_logs
    """).collect()[0]["max_time"]

    df_comments_inc = df_comments.filter(col("ingestion_time") > lit(max_time_c))

else:
    df_comments_inc = df_comments

In [0]:
#merge silver(posts)
if spark.catalog.tableExists("log_analytics.silver.posts_logs"):

    silver_posts = DeltaTable.forName(spark, "log_analytics.silver.posts_logs")

    silver_posts.alias("target").merge(
        df_posts_inc.alias("source"),
        "target.id = source.id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:

    df_posts_inc.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("log_analytics.silver.posts_logs")

In [0]:
#merge silver(comments)
if spark.catalog.tableExists("log_analytics.silver.comments_logs"):

    silver_comments = DeltaTable.forName(spark, "log_analytics.silver.comments_logs")

    silver_comments.alias("target").merge(
        df_comments_inc.alias("source"),
        "target.id = source.id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:

    df_comments_inc.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("log_analytics.silver.comments_logs")